### this notebook only works in colab to use it's gpu

In [ ]:
# ==============================================================================
# RUN THIS IN A GOOGLE COLAB NOTEBOOK WITH GPU ENABLED (Runtime > Change runtime type > T4 GPU)
# ==============================================================================

# Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart transformers torch pillow


In [ ]:
import io
import torch
import uvicorn
import nest_asyncio
from PIL import Image
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
# 1. Initialize FastAPI app
app = FastAPI(title="BLIP Captioning API")

# 2. Setup GPU and Load BLIP-1 Base model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BLIP model on device: {device}...")

model_id = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print("Model successfully loaded into memory!")

# 3. Define the captioning endpoint
@app.post("/caption")
async def generate_caption(file: UploadFile = File(...)):
    try:
        # Read uploaded image bytes
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # Preprocess and move to GPU
        inputs = processor(images=image, return_tensors="pt").to(
            device,
            dtype=torch.float16 if device == "cuda" else torch.float32
        )

        # Generate caption tokens
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=50)

        # Decode tokens to string
        caption = processor.decode(output_ids[0], skip_special_tokens=True)
        return JSONResponse(content={"caption": caption})

    except Exception as e:
        return JSONResponse(content={"error": str(e)}, status_code=500)

@app.get("/")
def health_check():
    return {"status": "BLIP service is active"}

Loading BLIP model on device: cuda...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Model successfully loaded into memory!


In [ ]:
import threading

NGROK_AUTH_TOKEN = ""
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Allow nested asyncio events in Colab
nest_asyncio.apply()

# Open a tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print("\n" + "=" * 60)
print(f"Public Colab API URL: {public_url}")
print("Copy this URL and paste it into your local Streamlit app!")
print("=" * 60 + "\n")

# Run the API server in a background thread to prevent asyncio runtime conflicts
def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()


Public Colab API URL: https://unclustering-irresistible-camille.ngrok-free.dev
Copy this URL and paste it into your local Streamlit app!

